# Data Preprocessing Pipeline

This notebook is the first step of the HDB resale data pipeline. It starts from the original raw resale CSV and performs the base preprocessing needed before feature enrichment.

Pipeline order:

`01_data_preprocessing_pipeline.ipynb` -> `02_feature_enrichment_pipeline.ipynb`

This step handles:

- flat model recategorization
- storey range conversion to a numeric midpoint
- raw-stage-safe cleanup
- log transformation of the response

Output:

- `data/hdb_resale_pipeline_intermediate.csv`


## Processing Conflicts And Resolution

There are a few differences across the source notebooks/scripts:

1. `test/cs3244_project.py` recategorizes `flat_model` into broader groups and converts `storey_range` to a numeric midpoint.
2. `hdb_resale_recategorised_cleaned.ipynb` assumes a later-stage enriched dataset already exists, so it drops columns such as `remaining_lease`, `mrt_name`, `month_date`, and `storey_mid`. Those columns are not all present in the original raw file, and some would be premature to drop at raw stage.
3. `Resale_price_data_analysis.ipynb` is mainly analysis, but it justifies creating `log_resale_price`.

Resolution used here:

- Apply flat-model grouping directly to the raw `flat_model` column.
- Convert `storey_range` from strings like `04 TO 06` into a numeric midpoint, and also create `storey_mid` for compatibility with downstream notebooks that assume it exists.
- Drop only raw-stage artifact columns such as unnamed index leftovers, and keep `block` / `street_name` because the downstream feature-addition step still uses them.
- Keep `lease_commence_date` and `remaining_lease`, because dropping them at this stage would remove useful raw information before later feature engineering.
- Add `log_resale_price` as the transformed response.

Important handoff note: `feature_engineering_hdb_resale.ipynb` still expects MRT/location-enriched inputs such as walking distance or coordinates. This notebook prepares the raw resale data only; another enrichment step is still needed before that notebook can run end-to-end.


In [2]:
from pathlib import Path
import re

import numpy as np
import pandas as pd


In [3]:
PROJECT_ROOT = Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "Resale Flat Prices from Jan 2015 to Feb 2026.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "hdb_resale_pipeline_intermediate.csv"

if not RAW_PATH.exists():
    raise FileNotFoundError("Could not find the original resale CSV in data/.")

raw_df = pd.read_csv(RAW_PATH, low_memory=False)
print(f"Loaded: {RAW_PATH}")
print(f"Rows: {len(raw_df):,}")
print(f"Columns: {list(raw_df.columns)}")


Loaded: C:\Users\Study\OneDrive - National University of Singapore\Desktop\Y3S2\CS3244\CS3244_group-3_ResalePriceModeling\data\Resale Flat Prices from Jan 2015 to Feb 2026.csv
Rows: 262,855
Columns: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price']


In [4]:
def recategorize_flat_model(value):
    value = str(value).strip()
    if value in {"Standard", "Improved", "Simplified", "2-room", "Apartment"}:
        return "Standard"
    if value in {"Model A", "Model A2", "Model A-Maisonette"}:
        return "Model A Series"
    if value in {"New Generation", "Premium Apartment", "Premium Apartment Loft", "Premium Maisonette"}:
        return "New Gen"
    if value in {"Maisonette", "Adjoined flat", "Improved-Maisonette", "Terrace", "Type S1", "Type S2"}:
        return "Maisonette"
    if value in {"DBSS", "Multi Generation", "3Gen"}:
        return "Special Schemes"
    return np.nan

def storey_midpoint(value):
    match = re.match(r"\s*(\d+)\s*TO\s*(\d+)\s*", str(value), flags=re.IGNORECASE)
    if not match:
        return np.nan
    lower = int(match.group(1))
    upper = int(match.group(2))
    return (lower + upper) / 2.0


In [5]:
processed_df = raw_df.copy()

# 1. Flat model feature engineering
processed_df["flat_model"] = processed_df["flat_model"].apply(recategorize_flat_model)

# 2. Storey range conversion to numerical midpoint
processed_df["storey_range"] = processed_df["storey_range"].apply(storey_midpoint)
processed_df["storey_mid"] = processed_df["storey_range"]

# 3. Log transformation of response
processed_df["resale_price"] = pd.to_numeric(processed_df["resale_price"], errors="coerce")
if (processed_df["resale_price"] <= 0).any():
    raise ValueError("resale_price contains non-positive values; log transform is invalid.")
processed_df["log_resale_price"] = np.log(processed_df["resale_price"])

print("Unmapped flat_model values:", int(processed_df["flat_model"].isna().sum()))
print("Missing numerical storey_range values:", int(processed_df["storey_range"].isna().sum()))
processed_df.head()

processed_df = raw_df.copy()

# Keep original flat model for testing purposes
processed_df["flat_model_raw"] = processed_df["flat_model"]

# 1. Flat model feature engineering
processed_df["flat_model"] = processed_df["flat_model"].apply(recategorize_flat_model)

# 2. Storey range conversion to numerical midpoint
processed_df["storey_range"] = processed_df["storey_range"].apply(storey_midpoint)
processed_df["storey_mid"] = processed_df["storey_range"]

# 3. Log transformation of response
processed_df["resale_price"] = pd.to_numeric(processed_df["resale_price"], errors="coerce")
if (processed_df["resale_price"] <= 0).any():
    raise ValueError("resale_price contains non-positive values; log transform is invalid.")
processed_df["log_resale_price"] = np.log(processed_df["resale_price"])

print("Unmapped flat_model values:", int(processed_df["flat_model"].isna().sum()))
print("Missing numerical storey_range values:", int(processed_df["storey_range"].isna().sum()))
processed_df.head()

Unmapped flat_model values: 0
Missing numerical storey_range values: 0
Unmapped flat_model values: 0
Missing numerical storey_range values: 0


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,flat_model_raw,storey_mid,log_resale_price
0,2015-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,8.0,60.0,Standard,1986,70,255000.0,Improved,8.0,12.449019
1,2015-01,ANG MO KIO,3 ROOM,541,ANG MO KIO AVE 10,2.0,68.0,New Gen,1981,65,275000.0,New Generation,2.0,12.524526
2,2015-01,ANG MO KIO,3 ROOM,163,ANG MO KIO AVE 4,2.0,69.0,New Gen,1980,64,285000.0,New Generation,2.0,12.560244
3,2015-01,ANG MO KIO,3 ROOM,446,ANG MO KIO AVE 10,2.0,68.0,New Gen,1979,63,290000.0,New Generation,2.0,12.577636
4,2015-01,ANG MO KIO,3 ROOM,557,ANG MO KIO AVE 10,8.0,68.0,New Gen,1980,64,290000.0,New Generation,8.0,12.577636


In [6]:
# 4. Drop raw-stage artifact columns only
drop_candidates = ["Unnamed: 0", "Unnamed: 0.1", "...1"]
processed_df = processed_df.drop(columns=[col for col in drop_candidates if col in processed_df.columns])

print("Final columns:")
print(processed_df.columns.tolist())
processed_df.head()


Final columns:
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price', 'flat_model_raw', 'storey_mid', 'log_resale_price']


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,flat_model_raw,storey_mid,log_resale_price
0,2015-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,8.0,60.0,Standard,1986,70,255000.0,Improved,8.0,12.449019
1,2015-01,ANG MO KIO,3 ROOM,541,ANG MO KIO AVE 10,2.0,68.0,New Gen,1981,65,275000.0,New Generation,2.0,12.524526
2,2015-01,ANG MO KIO,3 ROOM,163,ANG MO KIO AVE 4,2.0,69.0,New Gen,1980,64,285000.0,New Generation,2.0,12.560244
3,2015-01,ANG MO KIO,3 ROOM,446,ANG MO KIO AVE 10,2.0,68.0,New Gen,1979,63,290000.0,New Generation,2.0,12.577636
4,2015-01,ANG MO KIO,3 ROOM,557,ANG MO KIO AVE 10,8.0,68.0,New Gen,1980,64,290000.0,New Generation,8.0,12.577636


## Save Intermediate Output

This saved file is the handoff artifact between the two pipeline notebooks. It is a cleaned intermediate dataset, not the final modeling dataset.


In [7]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
processed_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved processed dataset to: {OUTPUT_PATH}")
print(f"Output rows: {len(processed_df):,}")
print(f"Output columns: {len(processed_df.columns)}")


Saved processed dataset to: C:\Users\Study\OneDrive - National University of Singapore\Desktop\Y3S2\CS3244\CS3244_group-3_ResalePriceModeling\data\hdb_resale_pipeline_intermediate.csv
Output rows: 262,855
Output columns: 14
